In [ ]:
! pip install -q transformers torch torchvision

In [ ]:
from transformers import pipeline
from PIL import Image
from deepface import DeepFace
import cv2

In [ ]:


def face_emotion(cap):
    print("Processing video... this may take a minute.")
    
    # Initialize dictionary to store the sum of emotion scores
    emotion_sums = {
        'angry': 0.0, 'disgust': 0.0, 'fear': 0.0, 
        'happy': 0.0, 'sad': 0.0, 'surprise': 0.0, 'neutral': 0.0
    }
    
    frame_count = 0
    analyzed_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Analyze every 10th frame to speed things up
        if frame_count % 10 == 0:
            try:
                # DeepFace.analyze returns a list (in case of multiple faces)
                # enforce_detection=False prevents the code from crashing if no face is found
                results = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)
                
                # Assume we are tracking the primary face (index 0)
                emotions = results[0]['emotion']
                
                for emotion, score in emotions.items():
                    if emotion in emotion_sums:
                        emotion_sums[emotion] += score
                
                analyzed_count += 1
                print ("analyzed_count:",analyzed_count)

            except Exception as e:
                print(f"Error occurred at frame {frame_count}: {e}")

        frame_count += 1

    cap.release()
    
    # Calculate average emotion scores
    if analyzed_count > 0:
        average_emotions = {emotion: (score / analyzed_count) for emotion, score in emotion_sums.items()}
    else:
        average_emotions = {emotion: 0.0 for emotion in emotion_sums}

    print("Done!")
    return average_emotions

In [ ]:
def face_emotion(cap):
    
    # 1. Load the model (Hugging Face pipelines don't use 'enforce_detection'—that's a DeepFace setting)
    classifier = pipeline("image-classification", model="abhilash88/face-emotion-detection")
    label_to_emotion = {
    'LABEL_0': 'angry',
    'LABEL_1': 'disgust',
    'LABEL_2': 'fear',
    'LABEL_3': 'happy',
    'LABEL_4': 'neutral',
    'LABEL_5': 'sad',
    'LABEL_6': 'surprise'
}
    print("Processing video... this may take a minute.")
    frame_count = 0
    # claculate average emotion for all frame using deepface and store it in a dictionary including angry, fear, neutral, sad, disgust, happy and surprise and each one label with 
    emotion_sums = {
    'angry': 0.0, 'disgust': 0.0, 'fear': 0.0, 
    'happy': 0.0, 'sad': 0.0, 'surprise': 0.0, 'neutral': 0.0
}
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
    #need to change
        # Analyze every 10th frame to speed things up
        if frame_count % 10 == 0:
            try:
                # IMPORTANT: OpenCV uses BGR, but Transformers/PIL use RGB.
                # Convert color and then convert the array to a PIL Image.
                img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_img = Image.fromarray(img_rgb)

                # Run the classifier on the PIL Image
                results = classifier(pil_img)
                # results is a list of dicts with 'label' and 'score'
                for res in results:
                    emotion_label = label_to_emotion.get(res['label'], 'unknown')
                    emotion_sums[emotion_label] += res['score']

            except Exception as e:
                print(f"Error occurred at frame {frame_count}: {e}")
                pass

        frame_count += 1

    cap.release()
    print("Done!")
    # Calculate average emotion scores
    total_frames_analyzed = frame_count // 10
    average_emotions = {emotion: (score / total_frames_analyzed) for emotion, score in emotion_sums.items()}
    return average_emotions

In [ ]:
input_video = "/content/Hossam_video.mp4"
cap = cv2.VideoCapture(input_video)
face_emotion(cap)

In [ ]:
from transformers import pipeline
from PIL import Image
import cv2

# 1. Load the model (Hugging Face pipelines don't use 'enforce_detection'—that's a DeepFace setting)
classifier = pipeline("image-classification", model="abhilash88/face-emotion-detection")

input_video = "/content/Hossam_video.mp4"
cap = cv2.VideoCapture(input_video)

print("Processing video... this may take a minute.")
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
#need to change
    # Analyze every 3rd frame to speed things up
    if frame_count % 10 == 0:
        try:
            # IMPORTANT: OpenCV uses BGR, but Transformers/PIL use RGB.
            # Convert color and then convert the array to a PIL Image.
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img_rgb)

            # Run the classifier on the PIL Image
            results = classifier(pil_img)
            # print each emothion with it's property from results
            for emotion_data in results:
                label = emotion_data['label']
                score = emotion_data['score']
                print(f"Emotion: {label}, Score: {score:.4f}")

        except Exception as e:
            print(f"Error occurred at frame {frame_count}: {e}")
            pass

    frame_count += 1

cap.release()
print("Done!")

In [ ]:
from transformers import pipeline

# Load the model one last time
pipe = pipeline("image-classification", model="abhilash88/face-emotion-detection")

# Save everything to a folder named 'my_emotion_model'
pipe.save_pretrained("my_emotion_model")